# Comparing Clustering Methods on Toy Datasets

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand** the fundamental differences between clustering algorithms
2. **Recognize** which clustering methods work best for different data shapes
3. **Apply** multiple clustering techniques using scikit-learn
4. **Evaluate** a partition with the silhouette score and WCSS, and say what these metrics do not tell you
5. **Identify** the strengths and weaknesses of each method

## Why Compare Clustering Methods?

In real-world data science, **no single clustering algorithm works best for all datasets**. Different algorithms make different assumptions about the structure of your data:

- **K-Means**: Assumes spherical clusters of similar size
- **Hierarchical (Agglomerative)**: Builds clusters incrementally; linkage method matters
- **DBSCAN**: Finds arbitrary-shaped clusters based on density

This notebook uses **synthetic toy datasets** with known structures to help you develop intuition about when each method succeeds or fails. Every dataset is generated with a fixed seed, so the figures you get are the ones the text describes.

## Datasets We'll Explore

We'll test four clustering methods on six different dataset shapes:

1. **Circles**: Concentric rings (non-convex shapes)
2. **Moons**: Crescent shapes (non-linear separability)
3. **Blobs**: Well-separated spherical clusters (ideal case)
4. **No Structure**: Uniform random points (no true clusters)
5. **Anisotropic**: Stretched/rotated clusters (non-spherical)
6. **Varied Variance**: Clusters with different densities

## Clustering Methods We'll Compare

For each dataset, we'll apply:

1. **K-Means** (partition-based)
2. **Agglomerative with Complete Linkage** (hierarchical)
3. **DBSCAN** (density-based)
4. **Agglomerative with Single Linkage** (hierarchical)

After each dataset we print a small table with the silhouette score, the WCSS, the number of clusters and the number of noise points for every method, and compare the numbers with the pictures.

## How Each Method Works: Intuitive Explanations

### 1. **K-Means** (Partition-Based)
**Think of it as**: Finding K "center points" and assigning each data point to the nearest center.

**How it works**:
- Start with K random center points (centroids)
- Assign each point to its nearest centroid
- Move centroids to the average position of their assigned points
- Repeat until centroids stop moving

**Key assumption**: Clusters are spherical (circular/ball-shaped) with similar sizes

**Best for**: Well-separated, roughly circular clusters of similar size

---

### 2. **Agglomerative Hierarchical - Complete Linkage** (Bottom-Up)
**Think of it as**: Starting with every point as its own cluster, then repeatedly merging the two "most similar" clusters.

**How it works**:
- Each point starts as a tiny cluster of size 1
- Find the two clusters with the smallest distance between their **farthest points** (complete linkage)
- Merge them into one cluster
- Repeat until you have K clusters

**Key characteristic**: Creates compact, spherical clusters by using the maximum distance

**Best for**: Finding well-defined, compact groups

---

### 3. **DBSCAN** (Density-Based Spatial Clustering)
**Think of it as**: Finding "crowded neighborhoods" of points and treating sparse areas as noise/outliers.

**How it works**:
- Pick a point and find all neighbors within radius `eps`
- If it has enough neighbors (`min_samples`), start a cluster
- Expand the cluster by including neighbors of neighbors
- Points in sparse regions are labeled as outliers

**Key characteristic**: Discovers clusters of arbitrary shapes; identifies outliers

**Best for**: Non-spherical clusters, noisy data, when you don't know K in advance

---

### 4. **Agglomerative Hierarchical - Single Linkage** (Bottom-Up)
**Think of it as**: Like complete linkage, but using the distance between **closest points** instead.

**How it works**:
- Each point starts as its own cluster
- Find the two clusters with the smallest distance between their **nearest points** (single linkage)
- Merge them
- Repeat until you have K clusters

**Key characteristic**: Can "chain" through elongated structures by connecting nearest neighbors

**Best for**: Elongated, non-spherical clusters; can struggle with clusters of different densities

**Warning**: Prone to "chaining effect" where clusters merge into one long chain

---

**Useful Resources:**

- [Scikit-learn Clustering Overview](https://scikit-learn.org/stable/modules/clustering.html)
- [K-Means Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
- [DBSCAN Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html)


## Step 1: Import Required Libraries

We'll use:
- **NumPy**: Numerical computations and array operations
- **Pandas**: Data manipulation and analysis
- **Matplotlib**: Data visualization
- **Scikit-learn**: Machine learning algorithms and synthetic datasets


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import datasets, metrics
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN

warnings.filterwarnings('ignore')


## Step 2: Create Synthetic Datasets

We'll generate six different synthetic datasets, each with 1,500 data points. These datasets represent different challenges for clustering algorithms. Every generator gets a fixed seed (`random_state=0`, or a seeded NumPy generator for the uniform points), so the pictures are reproducible.

### Dataset Descriptions:

1. **Circles (`make_circles`)**: Two concentric circles. Tests whether algorithms can handle non-convex shapes.
   - `factor=0.5`: Inner circle radius is 50% of outer circle
   - `noise=0.05`: Small random noise added to each point

2. **Moons (`make_moons`)**: Two interleaving half-circles. Tests non-linear cluster boundaries.

3. **Blobs (`make_blobs`)**: Well-separated spherical clusters. This is the "ideal" case for most algorithms.

4. **No Structure**: Uniformly distributed random points with no inherent clusters. Tests whether algorithms inappropriately force structure.

5. **Anisotropic**: Blobs that have been stretched and rotated via a transformation matrix. Tests sensitivity to cluster shape.

6. **Varied Variance**: Blobs with different densities/spreads (`cluster_std=[1.0, 2.5, 0.5]`). Tests handling of unequal cluster sizes.


In [ ]:
n_samples = 1500

# noise: standard deviation of the Gaussian noise added to each point; factor: inner/outer radius ratio
X_circles, y_circles = datasets.make_circles(n_samples=n_samples, factor=0.5, noise=0.05, random_state=0)
X_moons, y_moons = datasets.make_moons(n_samples=n_samples, noise=0.05, random_state=0)
X_blobs, y_blobs = datasets.make_blobs(n_samples=n_samples, random_state=8)

rng = np.random.default_rng(0)
X_no_structure, y_no_structure = rng.random((n_samples, 2)), None

# Anisotropic: blobs stretched and rotated by a linear transformation
random_state = 170
X_base, y_base = datasets.make_blobs(n_samples=n_samples, random_state=random_state)
transformation = [[0.6, -0.6], [-0.4, 0.8]]
X_aniso, y_aniso = np.dot(X_base, transformation), y_base

# Blobs with different spreads
X_varied, y_varied = datasets.make_blobs(n_samples=n_samples, cluster_std=[1.0, 2.5, 0.5], random_state=random_state)


## Step 3: Visualize the Datasets

Before applying clustering algorithms, let's visualize each dataset to understand their structure. This helps us develop intuition about which algorithms might work well.

**Look for:**
- How many natural clusters appear to exist?
- Are clusters spherical or irregular?
- Are clusters well-separated or overlapping?
- Are clusters of similar or different densities?


In [ ]:
def show(X, title='', label=None):
    '''Scatter plot of a 2-D dataset, coloured by label if one is given.'''
    plt.figure(figsize=(12, 6))
    if label is None:
        plt.scatter(X[:, 0], X[:, 1], color='steelblue', alpha=0.6, edgecolors='black', linewidth=0.5, s=50)
    else:
        plt.scatter(X[:, 0], X[:, 1], c=label, cmap='Accent', s=50)
    plt.title(title, fontsize=16)
    plt.show()


for name, X in [('Circles', X_circles), ('Moons', X_moons), ('Blobs', X_blobs),
                ('No structure (uniform)', X_no_structure), ('Anisotropic', X_aniso), ('Varied variance', X_varied)]:
    show(X, name)


## Step 4: Define Helper Functions

We'll create wrapper functions for each clustering method that return:
- **Silhouette Score**: Measures how well-separated clusters are (1 = best, 0 = overlapping, -1 = poor)
- **WCSS (Within-Cluster Sum of Squares)**: Total variance within clusters (lower is better)
- **Labels**: Cluster assignment for each point
- **Centroids**: Center points of each cluster

Two things to notice before you read the functions:

- WCSS is only defined relative to a centroid, so for the hierarchical methods and DBSCAN we compute it ourselves from the mean of each cluster (`eval_WCSS`). For k-means it is `kmeans.inertia_`.
- DBSCAN labels noise points as `-1`. They are left out of the silhouette and the WCSS, and counted separately. If DBSCAN finds fewer than two clusters the silhouette is undefined and we report `NaN`.

We also keep a small results log so that, after each dataset, we can print all four methods side by side.

### Function 1: Calculate Within-Cluster Sum of Squares (WCSS)

WCSS measures cluster compactness by summing squared distances from each point to its cluster centroid.


In [ ]:
def eval_WCSS(X, label):
    '''WCSS and centroids of a labelling; noise points (label -1) are ignored.'''
    df = pd.DataFrame(data=X, index=label)
    df = df[df.index != -1]
    centroids = df.groupby(level=0).mean()
    WCSS = 0.0
    for clust, centr in centroids.iterrows():
        points = df[df.index == clust].values
        WCSS += ((points - centr.values) ** 2).sum()
    return WCSS, centroids.values


### Function 2: K-Means Clustering

K-Means tries to minimize the distance between points and their assigned cluster centroids.


In [ ]:
def cluster_kmeans(df, nclust):
    kmeans = KMeans(n_clusters=nclust, n_init=10, random_state=0).fit(df)
    label = kmeans.labels_
    centroids = kmeans.cluster_centers_
    sil = metrics.silhouette_score(df, label, metric='euclidean', random_state=0)
    wcss = kmeans.inertia_
    return sil, wcss, label, centroids


### Function 3: Agglomerative Hierarchical Clustering

Agglomerative clustering starts with each point as its own cluster and iteratively merges the closest clusters.

**Linkage methods**:
- **Single linkage**: Distance between closest points in two clusters (can create "chains")
- **Complete linkage**: Distance between farthest points (creates more compact clusters)
- **Average linkage**: Average distance between all pairs of points


In [ ]:
def cluster_agglom(df, nclust, link):
    hc = AgglomerativeClustering(n_clusters=nclust, metric='euclidean', linkage=link).fit(df)
    label = hc.labels_
    wcss, centroids = eval_WCSS(df, label)
    sil = metrics.silhouette_score(df, label, metric='euclidean', random_state=0)
    return sil, wcss, label, centroids


### Function 4: DBSCAN

DBSCAN has no K. Its two parameters are the neighbourhood radius `eps` and the minimum number of points `min_samples` that a neighbourhood needs to start a cluster.


In [ ]:
def cluster_dbscan(df, eps, min_samples=5):
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(df)
    label = db.labels_
    core = label != -1
    n_clusters = len(set(label[core]))
    if n_clusters >= 2:
        sil = metrics.silhouette_score(df[core], label[core], metric='euclidean', random_state=0)
    else:
        sil = np.nan
    wcss, centroids = eval_WCSS(df, label)
    return sil, wcss, label, centroids


### Results log

`record` stores one row per (dataset, method); `metrics_table` prints the rows of one dataset.


In [ ]:
results = []

def record(dataset_name, method, sil, wcss, label):
    results.append({'dataset': dataset_name, 'method': method, 'silhouette': sil, 'WCSS': wcss,
                    'n_clusters': len(set(label)) - (1 if -1 in label else 0), 'n_noise': int((label == -1).sum())})

def metrics_table(dataset_name):
    df = pd.DataFrame([r for r in results if r['dataset'] == dataset_name]).drop(columns='dataset').set_index('method')
    display(df.round(3))


---

# Dataset 1: Concentric Circles

## Challenge: Non-Convex Clusters

The circles dataset contains two concentric rings. This tests whether algorithms can identify non-convex (non-spherical) cluster shapes.

**Expected Results:**
- **K-Means**: Will likely fail because it assumes spherical clusters
- **Complete Linkage**: May struggle due to spherical bias
- **DBSCAN**: Should succeed if epsilon is tuned correctly
- **Single Linkage**: Can follow the ring structure ("chaining effect")

Let's see if our predictions are correct!

### Method 1: K-Means Clustering

K-Means will try to split the data into 2 clusters by finding centroids that minimize within-cluster variance.


In [ ]:
sil, wcss, label, centroids = cluster_kmeans(X_circles, 2)
record('circles', 'k-means', sil, wcss, label)
show(X_circles, 'Circles: k-means, K=2', label)


**Observation**: K-means cuts the two rings down the middle. Each cluster holds half of the inner ring and half of the outer ring, because every point goes to the nearest centroid and that boundary is always a straight line. The rings themselves are never separated.


### Method 2: Agglomerative Clustering (Complete Linkage)

Complete linkage merges clusters based on the maximum distance between points.


In [ ]:
sil, wcss, label, centroids = cluster_agglom(X_circles, 2, link='complete')
record('circles', 'complete linkage', sil, wcss, label)
show(X_circles, 'Circles: complete linkage, 2 clusters', label)


**Observation**: Complete linkage also fails. It merges compact groups, and the most compact two-way split of two rings is again a cut across both rings, here roughly the upper part of both rings against the rest. The result looks like k-means with a tilted boundary.


### Method 3: DBSCAN (Density-Based)

DBSCAN groups points that are close together (density-connected) without assuming cluster shape.

**Key parameter**: `eps` defines the neighborhood radius for each point. We use `eps=0.15` here; the observation below explains why 0.2 is too large for these rings.


In [ ]:
sil, wcss, label, centroids = cluster_dbscan(X_circles, eps=0.15)
record('circles', 'DBSCAN (eps=0.15)', sil, wcss, label)
show(X_circles, 'Circles: DBSCAN, eps=0.15', label)


**Observation**: With `eps=0.15` DBSCAN recovers the two rings exactly (750 points each in this run, no noise). The radius matters: the gap between the rings is about 0.13 in this data, so `eps=0.2` would connect the rings through the noisiest points and return one cluster. Change `eps` to 0.2 in the cell above and rerun to see that happen.


### Method 4: Agglomerative Clustering (Single Linkage)

Single linkage merges clusters based on the minimum distance between points.


In [ ]:
sil, wcss, label, centroids = cluster_agglom(X_circles, 2, link='single')
record('circles', 'single linkage', sil, wcss, label)
show(X_circles, 'Circles: single linkage, 2 clusters', label)


**Observation**: Single linkage separates the two rings perfectly in this run (750 and 750 points). It keeps merging nearest neighbours, so it walks around each ring before it ever has to jump the gap between them. It works here because the gap is wider than any nearest-neighbour distance inside a ring; one bridging point would have been enough to merge them.


In [ ]:
metrics_table('circles')


### Summary: Circles Dataset

| Method | What happened (this run) |
|--------|--------------------------|
| K-Means | Cut both rings in half with a straight boundary |
| Complete Linkage | Cut both rings with a tilted boundary |
| DBSCAN (eps=0.15) | Found the two rings exactly; eps=0.2 would have merged them |
| Single Linkage | Found the two rings exactly |

**Read the metrics table before you trust it.** The two methods that got the rings right have the *lowest* silhouette (about 0.11 in this run), and k-means, which got them wrong, has the highest (about 0.36). The silhouette rewards compact, well-separated groups; a ring is not compact, since points on opposite sides of the same ring are far apart. WCSS is even less useful here: the correct two-ring partition and a single cluster containing everything have the same WCSS (about 940), because both rings have their centroid at the origin.

**Key insight**: For non-convex shapes, use density-based (DBSCAN) or single-linkage methods, and judge the result by looking at it. Silhouette and WCSS measure compactness, not correctness.


---

# Dataset 2: Moons (Crescent Shapes)

## Challenge: Non-Linear Separability

The moons dataset contains two interleaving crescent shapes. No straight line can separate them.

**Expected Results:** Similar to circles, density-based methods should excel.

### Method 1: K-Means


In [ ]:
sil, wcss, label, centroids = cluster_kmeans(X_moons, 2)
record('moons', 'k-means', sil, wcss, label)
show(X_moons, 'Moons: k-means, K=2', label)


**Observation**: K-means draws a slanted straight boundary through the middle: the left cluster takes most of the upper moon plus the left tail of the lower moon, the right cluster takes the tip of the upper moon plus most of the lower moon. In this run about a quarter of the points end up with the wrong moon.


### Method 2: Agglomerative (Complete Linkage)


In [ ]:
sil, wcss, label, centroids = cluster_agglom(X_moons, 2, link='complete')
record('moons', 'complete linkage', sil, wcss, label)
show(X_moons, 'Moons: complete linkage, 2 clusters', label)


**Observation**: Complete linkage does not fix this. One cluster is the entire upper moon *plus* the left part of the lower moon (957 against 543 points in this run), so the boundary still crosses a crescent. It is a different mistake from k-means, not a smaller one.


### Method 3: DBSCAN


In [ ]:
sil, wcss, label, centroids = cluster_dbscan(X_moons, eps=0.2)
record('moons', 'DBSCAN (eps=0.2)', sil, wcss, label)
show(X_moons, 'Moons: DBSCAN, eps=0.2', label)


**Observation**: DBSCAN with `eps=0.2` separates the two crescents exactly (750 and 750, no noise). It follows the dense band of each moon and never has to cross the gap between them.


### Method 4: Agglomerative (Single Linkage)


In [ ]:
sil, wcss, label, centroids = cluster_agglom(X_moons, 2, link='single')
record('moons', 'single linkage', sil, wcss, label)
show(X_moons, 'Moons: single linkage, 2 clusters', label)


**Observation**: Single linkage also separates the moons exactly, by the same chaining logic as on the circles.


In [ ]:
metrics_table('moons')


### Summary: Moons Dataset

| Method | What happened (this run) |
|--------|--------------------------|
| K-Means | Slanted straight cut; about a quarter of the points with the wrong moon |
| Complete Linkage | Whole upper moon plus part of the lower moon in one cluster |
| DBSCAN (eps=0.2) | Both crescents recovered exactly |
| Single Linkage | Both crescents recovered exactly |

**The silhouette again rewards the wrong answer.** In this run k-means has the highest silhouette (about 0.49) and the two correct partitions the lowest (about 0.34). K-means's straight cut produces two compact half-discs, which is what the silhouette measures; the crescents are correct but long and thin.

**Key insight**: Non-linear boundaries require density-based or single-linkage methods. When the metric and the picture disagree, the picture wins on toy data; on real data in more than two dimensions you do not have the picture, which is why the next notebook spends so much time on interpreting clusters.


---

# Dataset 3: Well-Separated Blobs

## Challenge: None (Ideal Case)

This dataset has well-separated, spherical clusters. This is the **ideal scenario** for most clustering algorithms.

**Expected Results:** All methods should perform well.

### Method 1: K-Means


In [ ]:
sil, wcss, label, centroids = cluster_kmeans(X_blobs, 3)
record('blobs', 'k-means', sil, wcss, label)
show(X_blobs, 'Blobs: k-means, K=3', label)


**Observation**: K-means recovers the three blobs. This is the ideal scenario: well-separated, spherical clusters of equal size match its assumptions exactly.


### Method 2: Agglomerative (Complete Linkage)


In [ ]:
sil, wcss, label, centroids = cluster_agglom(X_blobs, 3, link='complete')
record('blobs', 'complete linkage', sil, wcss, label)
show(X_blobs, 'Blobs: complete linkage, 3 clusters', label)


**Observation**: Complete linkage returns the identical partition (same silhouette and WCSS as k-means in the table below).


### Method 3: DBSCAN


In [ ]:
sil, wcss, label, centroids = cluster_dbscan(X_blobs, eps=0.5)
record('blobs', 'DBSCAN (eps=0.5)', sil, wcss, label)
show(X_blobs, 'Blobs: DBSCAN, eps=0.5', label)


**Observation**: DBSCAN with `eps=0.5` finds the three blobs and additionally labels 35 points on their outer edges as noise (in this run). That is not an error: the edge of a Gaussian blob really is sparser than its core. But it shows that DBSCAN's answer depends on `eps`: smaller and the blobs fragment, larger and the noise disappears, larger still and the two right-hand blobs merge.


### Method 4: Agglomerative (Single Linkage)


In [ ]:
sil, wcss, label, centroids = cluster_agglom(X_blobs, 3, link='single')
record('blobs', 'single linkage', sil, wcss, label)
show(X_blobs, 'Blobs: single linkage, 3 clusters', label)


**Observation**: Single linkage also returns the identical partition. With gaps this wide there is nothing to chain across.


In [ ]:
metrics_table('blobs')


### Summary: Blobs Dataset

| Method | What happened (this run) |
|--------|--------------------------|
| K-Means | Three blobs recovered |
| Complete Linkage | Identical partition to k-means |
| DBSCAN (eps=0.5) | Three blobs plus 35 edge points labelled noise |
| Single Linkage | Identical partition to k-means |

Here the metrics and the picture agree: silhouette about 0.83 for all three partition methods, and slightly higher for DBSCAN only because its noise points (the hardest ones) are excluded from the score.

**Key insight**: For well-separated spherical clusters, every method works and k-means is the fastest. Differences between methods only show up when the data violate k-means's assumptions.


---

# Dataset 4: No Structure (Random Uniform)

## Challenge: No True Clusters Exist

This dataset is uniformly distributed random noise with **no inherent cluster structure**.

**Expected Results:**
- Partition-based methods (K-Means) will **force** structure where none exists
- DBSCAN has no K, so it might behave differently. Before running it, write down what you expect: many noise points, one big cluster, or many small ones?

This tests whether algorithms inappropriately impose structure on structureless data.

### Method 1: K-Means


In [ ]:
sil, wcss, label, centroids = cluster_kmeans(X_no_structure, 3)
record('no structure', 'k-means', sil, wcss, label)
show(X_no_structure, 'No structure: k-means, K=3', label)


**Observation**: K-means returns three wedge-shaped regions that meet near the middle of the square. There is no structure in the data, but k-means **always returns exactly K clusters**; it partitions the square the way it would partition anything. Note the silhouette in the table below: about 0.38 in this run, higher than the silhouette of the correct answer on the circles. A positive silhouette does not mean clusters exist.


### Method 2: Agglomerative (Complete Linkage)


In [ ]:
sil, wcss, label, centroids = cluster_agglom(X_no_structure, 3, link='complete')
record('no structure', 'complete linkage', sil, wcss, label)
show(X_no_structure, 'No structure: complete linkage, 3 clusters', label)


**Observation**: Complete linkage also carves the square into three arbitrary regions (a band along the top, one bottom-left, one bottom-right in this run). Any method that is told to return three clusters will return three clusters.


### Method 3: DBSCAN

With 1,500 points on the unit square, a circle of radius `eps=0.1` contains on average 1500 x pi x 0.01, about 47 points, far more than `min_samples=5`. So every point is a core point and DBSCAN will connect everything. The cell also runs a small sweep over `eps` and prints the number of clusters and noise points for each value, so you can see what happens when the radius holds only a handful of points.


In [ ]:
sil, wcss, label, centroids = cluster_dbscan(X_no_structure, eps=0.1)
record('no structure', 'DBSCAN (eps=0.1)', sil, wcss, label)
show(X_no_structure, 'No structure: DBSCAN, eps=0.1', label)

print('eps sweep on the uniform data (min_samples=5):')
for eps in [0.02, 0.03, 0.05, 0.1]:
    lab = DBSCAN(eps=eps, min_samples=5).fit(X_no_structure).labels_
    expected = n_samples * np.pi * eps ** 2
    print(f'  eps={eps:<5} expected points per neighbourhood={expected:5.1f}  '
          f'clusters={len(set(lab)) - (1 if -1 in lab else 0):4d}  noise={int((lab == -1).sum()):5d}')


**Observation**: With `eps=0.1` DBSCAN returns **one cluster and no noise**. A uniform square is one connected region of constant density, and 47 expected neighbours per radius makes every point a core point, so DBSCAN correctly reports "everything is connected". It does not report "no clusters", because a single dense region *is* a cluster to DBSCAN.

The sweep printed under the figure shows what happens as `eps` shrinks. At `eps=0.05` (about 12 neighbours) it is still one cluster. At `eps=0.03` (about 4 neighbours, right at `min_samples`) the square fragments into some 70 small clusters with a few hundred noise points; at `eps=0.02` most points are noise. There is no `eps` at which DBSCAN finds a stable, small number of clusters, and that instability is the real signal that the data have no structure. DBSCAN does not need K, but the number of clusters it returns still depends on a parameter that has to be chosen relative to the density of the data.


### Method 4: Agglomerative (Single Linkage)


In [ ]:
sil, wcss, label, centroids = cluster_agglom(X_no_structure, 2, link='single')
record('no structure', 'single linkage', sil, wcss, label)
show(X_no_structure, 'No structure: single linkage, 2 clusters', label)


**Observation**: Asked for two clusters, single linkage splits off a single isolated point (1,499 against 1 in this run) and calls the rest one cluster. On structureless data the most isolated point is always the first thing single linkage separates.


In [ ]:
metrics_table('no structure')


### Summary: No Structure Dataset

| Method | What happened (this run) |
|--------|--------------------------|
| K-Means | Three wedge-shaped regions; silhouette about 0.38 |
| Complete Linkage | Three arbitrary regions; silhouette about 0.27 |
| DBSCAN (eps=0.1) | One cluster, no noise; fragments into dozens of clusters only when eps is at the min_samples limit |
| Single Linkage | One point split off, everything else one cluster |

**Key insight**: Always check whether clusters are meaningful. Every method here produced an answer, and the silhouette of the k-means answer would pass many rule-of-thumb thresholds. The checks that help are: compare the silhouette against the value obtained on shuffled or uniform data of the same size, look at whether the partition is stable under a different seed or a different `eps`, and ask whether the clusters mean anything in the domain.


---

# Dataset 5: Anisotropic (Stretched) Clusters

## Challenge: Non-Spherical Cluster Shapes

These clusters have been stretched and rotated by a transformation matrix, creating elongated (elliptical) shapes.

**Expected Results:**
- K-Means may struggle due to spherical assumption
- Hierarchical methods may adapt better

### Method 1: K-Means


In [ ]:
sil, wcss, label, centroids = cluster_kmeans(X_aniso, 3)
record('anisotropic', 'k-means', sil, wcss, label)
show(X_aniso, 'Anisotropic: k-means, K=3', label)


**Observation**: The right-hand stripe is recovered on its own, but the two parallel stripes on the left are cut *across* their length rather than separated from each other: one cluster is the upper-left ends of both stripes, the other their lower-right ends. K-means measures distance in all directions equally, and the two stripes are closer to each other side by side than their own two ends are. Roughly 490 to 520 points per cluster in this run, matching the true 500s only by accident.


### Method 2: Agglomerative (Complete Linkage)


In [ ]:
sil, wcss, label, centroids = cluster_agglom(X_aniso, 3, link='complete')
record('anisotropic', 'complete linkage', sil, wcss, label)
show(X_aniso, 'Anisotropic: complete linkage, 3 clusters', label)


**Observation**: Complete linkage makes the same kind of mistake with a different boundary: the two left stripes are merged over most of their length (760 points in one cluster in this run), and a middle segment, made of the lower ends of the two left stripes and the upper end of the right stripe, is split off as the third cluster (381 points), so the right stripe is cut as well. It does not follow the diagonal elongation any better than k-means; it also wants compact groups.


### Method 3: DBSCAN


In [ ]:
sil, wcss, label, centroids = cluster_dbscan(X_aniso, eps=0.3)
record('anisotropic', 'DBSCAN (eps=0.3)', sil, wcss, label)
show(X_aniso, 'Anisotropic: DBSCAN, eps=0.3', label)


**Observation**: DBSCAN with `eps=0.3` comes closest: the three stripes come out as three clusters of about 490 to 500 points each, with a fourth tiny cluster of 6 points and 16 noise points at the sparse ends of the stripes in this run. Density along a stripe is roughly constant, so density-based growth follows the stripe. The small extra cluster and the noise are the price of a fixed radius on data whose tails thin out.


### Method 4: Agglomerative (Single Linkage)


In [ ]:
sil, wcss, label, centroids = cluster_agglom(X_aniso, 3, link='single')
record('anisotropic', 'single linkage', sil, wcss, label)
show(X_aniso, 'Anisotropic: single linkage, 3 clusters', label)


**Observation**: Single linkage fails by chaining. The three stripes are close enough to each other that nearest-neighbour merging connects them long before it runs out of within-stripe links, so almost everything ends up in one cluster (1,498 points in this run) and the remaining two clusters are two isolated outlier points at the far ends. The silhouette (about 0.21) and the WCSS (six times k-means's) both flag this, for once correctly.


In [ ]:
metrics_table('anisotropic')


### Summary: Anisotropic Dataset

| Method | What happened (this run) |
|--------|--------------------------|
| K-Means | Right stripe correct; the two left stripes cut across their length |
| Complete Linkage | Two left stripes merged; a middle segment (lower ends of the left stripes plus the upper end of the right stripe) split off |
| DBSCAN (eps=0.3) | Three stripes recovered, plus a 6-point fragment and 16 noise points |
| Single Linkage | One giant chained cluster plus two single outlier points |

The silhouette ranks k-means first (about 0.51) and DBSCAN third (about 0.41), the reverse of what the pictures show.

**Key insight**: Elongated clusters break the "distance to centre" logic of k-means and complete linkage. Density-based methods follow the elongation. Single linkage would too, but only if the stripes were further apart than the spacing of points within a stripe; here they are not, and it chains. Rescaling or rotating the features (or a Gaussian mixture with full covariances) is the usual fix for k-means.


---

# Dataset 6: Varied Variance (Different Densities)

## Challenge: Unequal Cluster Sizes and Densities

This dataset has three clusters with very different spreads:
- Cluster 1: `std = 1.0` (tight)
- Cluster 2: `std = 2.5` (loose)
- Cluster 3: `std = 0.5` (very tight)

**Expected Results:**
- K-Means assumes equal variance
- DBSCAN uses a single `eps` parameter for all clusters

### Method 1: K-Means


In [ ]:
sil, wcss, label, centroids = cluster_kmeans(X_varied, 3)
record('varied variance', 'k-means', sil, wcss, label)
show(X_varied, 'Varied variance: k-means, K=3', label)


**Observation**: K-means recovers the three groups reasonably well in this run: the two tight blobs on the left and right, and the loose cluster in the middle. Its one systematic error is at the boundary with the right-hand blob, where points from the loose cluster that drift close to the tight blob are assigned to it (the clusters come out as about 565, 530 and 405 points against a true 500 each). K-means assumes equal spread, so it draws the boundary halfway between centroids instead of closer to the tight blob.


### Method 2: Agglomerative (Complete Linkage)


In [ ]:
sil, wcss, label, centroids = cluster_agglom(X_varied, 3, link='complete')
record('varied variance', 'complete linkage', sil, wcss, label)
show(X_varied, 'Varied variance: complete linkage, 3 clusters', label)


**Observation**: Complete linkage does worse, not better. It cuts the loose middle cluster into pieces and attaches them to the tight blobs: one cluster is the tight left blob plus the lower half of the loose cluster (784 points in this run), one is the tight right blob plus the right edge of the loose cluster, and the third is a small remnant of 114 points at the top of the loose cluster. Complete linkage merges by the *farthest* pair of points, so a wide, loose group is expensive to keep together and gets broken up.


### Method 3: DBSCAN


In [ ]:
sil, wcss, label, centroids = cluster_dbscan(X_varied, eps=0.6)
record('varied variance', 'DBSCAN (eps=0.6)', sil, wcss, label)
show(X_varied, 'Varied variance: DBSCAN, eps=0.6', label)


**Observation**: DBSCAN with `eps=0.6` finds the two tight blobs cleanly and the dense core of the loose cluster, but the loose cluster's outskirts become noise (68 points in this run) and three small fragments (17, 5 and 4 points). One `eps` cannot fit a cluster with standard deviation 0.5 and one with 2.5 at the same time: a radius that holds the tight blobs together is too small for the sparse edges of the loose one.


### Method 4: Agglomerative (Single Linkage)


In [ ]:
sil, wcss, label, centroids = cluster_agglom(X_varied, 3, link='single')
record('varied variance', 'single linkage', sil, wcss, label)
show(X_varied, 'Varied variance: single linkage, 3 clusters', label)


**Observation**: Single linkage chains everything into one cluster (1,496 points in this run) and spends the other two clusters on two pairs of outlier points, one on the far left and one above the loose cluster. The loose cluster touches both tight blobs, so nearest-neighbour merging never finds a gap to stop at. The silhouette (about 0.11) and the WCSS (eight times k-means's) both reflect the failure.


In [ ]:
metrics_table('varied variance')


### Summary: Varied Variance Dataset

| Method | What happened (this run) |
|--------|--------------------------|
| K-Means | Three groups recovered; boundary with the tight right blob slightly off |
| Complete Linkage | Loose cluster broken up and attached to the tight blobs |
| DBSCAN (eps=0.6) | Tight blobs and the loose core found; the loose edges become noise and fragments |
| Single Linkage | One chained cluster plus two pairs of outliers |

Here the silhouette ordering (k-means 0.65, complete 0.51, DBSCAN 0.49, single 0.11) happens to match the pictures.

**Key insight**: Unequal densities hurt every method, but differently. K-means misplaces the boundary; complete linkage dismembers the loose cluster; DBSCAN cannot serve two densities with one `eps`; single linkage chains. If clusters are known to differ in spread, a Gaussian mixture model (which estimates a spread per cluster) is the natural tool.


In [ ]:
# All results in one table
all_results = pd.DataFrame(results).set_index(['dataset', 'method'])
display(all_results.round(3))


---

# Overall Summary & Recommendations

## What the six datasets showed

| Dataset | K-Means | Complete linkage | DBSCAN | Single linkage |
|---|---|---|---|---|
| Circles | cut across rings | cut across rings | correct (eps tuned) | correct |
| Moons | straight cut | crescent plus part of the other | correct | correct |
| Blobs | correct | correct | correct plus edge noise | correct |
| No structure | 3 wedges | 3 regions | one cluster | one point split off |
| Anisotropic | cut across stripes | stripes merged | close to correct | chained |
| Varied variance | close to correct | loose cluster dismembered | cores found, edges noise | chained |

## What the metrics showed

- The **silhouette** rewarded k-means's wrong split of the circles and the moons over the correct ring and crescent partitions, gave a respectable 0.38 to three wedges in uniform noise, and ranked k-means above DBSCAN on the stripes. It measures compactness and separation of whatever partition it is given.
- **WCSS** could not distinguish the correct two-ring partition from a single cluster on the circles (same centroid), and always falls with K. It is useful for comparing K within one method (the elbow), not for comparing methods or for deciding whether clusters exist.
- Both metrics did flag the chaining failures of single linkage, where one giant cluster and a few outliers really are a bad partition by any measure.

**Rule**: use the metrics to compare settings of one method, and the pictures (or, in more dimensions, the interpretation of the clusters) to decide whether the method is the right one.

## When to Use Each Method

### Use K-Means When:
- Clusters are roughly **spherical** and of **similar spread**
- You need **fast computation** on large datasets
- You can choose the **number of clusters** with a K sweep

### Use Hierarchical Clustering When:
- You want to explore **multiple clustering granularities** with a dendrogram
- The dataset is **small to medium-sized** (the algorithm is O(n squared))

**Linkage Choice:**
- **Single linkage**: follows elongated or ring-shaped clusters if they are clearly separated; chains as soon as clusters touch
- **Complete linkage**: compact clusters; breaks up wide, loose groups
- **Average linkage**: a compromise between the two

### Use DBSCAN When:
- Clusters have **arbitrary shapes** (non-convex)
- You want to **identify outliers/noise**
- The number of clusters is **unknown**
- Clusters have **similar densities**

**Caution**: `eps` must be chosen relative to the density of the data (a k-distance plot is the usual tool); one `eps` cannot serve clusters of very different density.

---

## Key Takeaways

1. **No universal best method**: the choice depends on the shape and density of the clusters.
2. **Assumptions matter**: k-means assumes spherical, equal-spread clusters, and fails in predictable ways when they are violated.
3. **Every method returns an answer**, including on data with no structure. Validation is your job.
4. **Metrics are not oracles**: silhouette and WCSS measure compactness, and can prefer a wrong partition.
5. **Parameters matter**: DBSCAN's result changes qualitatively with `eps`; hierarchical results change with the linkage.

---

## Financial Applications

In finance and banking, clustering is used for:

- **Customer segmentation**: group customers by behaviour (k-means is the usual first choice)
- **Credit risk**: identify borrower groups with similar risk profiles (see the next notebook)
- **Fraud detection**: DBSCAN's noise label is a natural anomaly flag
- **Portfolio construction**: cluster assets by return patterns
- **Market regime detection**: identify market states (calm, volatile, crisis)

---

## Further Reading

- [Scikit-learn Clustering Comparison](https://scikit-learn.org/stable/auto_examples/cluster/plot_cluster_comparison.html), the example this notebook is modelled on
- [Choosing the Right Clustering Method](https://scikit-learn.org/stable/modules/clustering.html#overview-of-clustering-methods)

---

## Practice Exercises

1. **DBSCAN on the varied-variance data.** Run `cluster_dbscan(X_varied, eps)` for `eps` in {0.3, 0.6, 1.0, 1.5} and record clusters and noise points. Deliverable: a four-row table and one sentence on which `eps` you would choose and what it gets wrong.
2. **A silhouette baseline.** Compute the k-means silhouette for K = 2 to 6 on `X_no_structure` and on `X_blobs`. Deliverable: the two series of five numbers and one sentence on what silhouette value you would need to see on real data before believing that clusters exist.
3. **Your own data.** Take a real financial dataset with at least two numeric features (for example daily returns and volatility of a set of stocks, or the ESG dataset in `data/company_esg_financial_dataset.csv`), standardise the features, and run all four methods. Deliverable: the metrics table, a scatter plot of the partition you find most interpretable, and one paragraph explaining your choice of method to a non-technical colleague.
